In [1]:
from __future__ import print_function

import os


os.chdir("../")

In [2]:
from hyperopt import STATUS_OK, Trials, tpe
from hyperas.distributions import choice, uniform, loguniform
from hyperas import optim

import tensorflow as tf
from tensorflow.keras import models, optimizers
from tensorflow.keras.datasets import cifar10

from sklearn.model_selection import train_test_split

import numpy as np

from autoencoder.variational_autoencoder import VariationalAutoencoder
from common.utils import (get_compile_args, get_callbacks, 
                        hyperas_path, i, best_acc, save_logs, 
                        init, load_samples, load_cifar10)

In [3]:
def create_data():
    init()

    feature_train0, y_train, *_ = load_cifar10(return_features=True, onehot_labels=True, verbose=0)
    feature_train1, *_ = load_cifar10(return_features=True, preprocess="normalize", verbose=0)
    feature_train2, *_ = load_cifar10(return_features=True, preprocess="min-max", verbose=0)

    clf0 = models.load_model("./models/hyperas/cifar10_dnn_model_00.h5")
    clf1 = models.load_model("./models/hyperas/cifar10_dnn_model_00B.h5")
    clf2 = models.load_model("./models/hyperas/cifar10_dnn_model_00C.h5")

    return feature_train0, feature_train1, feature_train2, y_train, clf0, clf1, clf2

In [4]:
def create_model(feature_train0, feature_train1, feature_train2, y_train, clf0, clf1, clf2):
    global best_acc, i  


    hidden_num = {{choice(["one", "two", "three", "four"])}}
    units1 = {{choice([16, 32, 64, 128, 256, 512, 1024])}}
    units2 = int(units1 // {{choice([1, 2, 4])}})
    units3 = int(units2 // {{choice([1, 2, 4])}})
    units4 = int(units3 // {{choice([1, 2, 4])}})
    latent_dim = int(units4 // {{choice([1, 2, 4])}})

    actv = {{choice(["relu", "elu", "selu", "prelu", "tanh", "sigmoid"])}}
    last_actv = {{choice(["linear", "tanh", "sigmoid"])}}
    use_batch_norm = {{choice([True, False])}}

    distribution = {{choice(["normal", "uniform"])}}
    initialization = {{choice(["glorot", "he", "lecun"])}}
    kernel_init = f"{initialization}_{distribution}"

    lr = {{choice([0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1])}}
    momentum = {{choice([0.0, 0.01, 0.03, 0.1, 0.3, 0.5, 0.8, 0.9, 0.99])}}
    weight_decay = {{choice([0.0, 0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1])}}
    opt_choice = {{choice(["sgd", "rmsprop", "adam", "nadam", "adamax", "adamw"])}}

    loss_choice = {{choice(["bce", "mae", "mse"])}}

    beta = {{choice([0.1, 0.25, 0.5, 0.75, 1., 1.25, 1.5, 1.75, 2.])}}

    sgd = optimizers.SGD(learning_rate=lr, momentum=momentum, decay=weight_decay, nesterov=True)
    rmsprop = optimizers.RMSprop(learning_rate=lr, momentum=momentum, decay=weight_decay)
    adam = optimizers.Adam(learning_rate=lr, decay=weight_decay)
    nadam = optimizers.Nadam(learning_rate=lr, decay=weight_decay)
    adamax = optimizers.Adamax(learning_rate=lr, decay=weight_decay)
    adamw = optimizers.experimental.AdamW(learning_rate=lr, weight_decay=weight_decay)

    if opt_choice == "sgd":
        optimizer = sgd
    elif opt_choice == "rmsprop":
        optimizer = rmsprop
    elif opt_choice == "adam":
        optimizer = adam
    elif opt_choice == "nadam":
        optimizer = nadam
    elif opt_choice == "adamax":
        optimizer = adamax
    elif opt_choice == "adamw":
        optimizer = adamw

    if loss_choice == "bce":
        loss_fn = "binary_crossentropy"
    elif loss_choice == "mae":
        loss_fn = "mean_absolute_error"
    elif loss_choice == "mse":
        loss_fn = "mean_squared_error"

    if last_actv == "linear":
        feature_train = feature_train0
        clf = clf0
    elif last_actv == "tanh":
        feature_train = feature_train1
        clf = clf1
    elif last_actv == "sigmoid":
        feature_train = feature_train2
        clf = clf2


    model = VariationalAutoencoder(
        conditioned=True, 
        class_num=10,
        latent_dim=latent_dim,
        hiddens_dims=[units1, units2, units3, units4],
        hiddens_kwargs={
            "actv": actv,
            "use_batch_norm": use_batch_norm,
            "kernel_init": kernel_init
        },
        last_activation=last_actv, 
        beta=beta,
        compile_args=get_compile_args(
            optimizer=optimizer,
            loss=loss_fn
        ),
        name="cifar10_vae_00"
    )

    history = model.train(
        feature_train, y_train,
        train_num=-1,
        epochs=50,
        batch_size=256,
        callbacks_list=get_callbacks(monitor="decoder_accuracy", verbose=0),
        clf=clf,
        verbose=0
    )

    val_acc = max(history["decoder_accuracy"])


    if val_acc > best_acc:
        best_acc = val_acc
        model.save(os.path.join(hyperas_path, f"{model.name}.tf"))

    save_logs(model.name, i, val_acc, best_acc, 
            search_space=[hidden_num, units1, units2, units3, units4, latent_dim, actv, last_actv, 
                        use_batch_norm, distribution, initialization, kernel_init, lr, momentum, 
                        weight_decay, opt_choice, loss_choice, preprocess_choice, beta], 
            names=["hidden_num", "units1", "units2", "units3", "units4", "latent_dim", "actv", "last_actv", 
                "use_batch_norm", "distribution", "initialization", "kernel_init", "lr", "momentum", 
                "weight_decay", "opt_choice", "loss_choice", "preprocess_choice", "beta"], 
            where_to="file")

    i += 1


    return {"loss": -val_acc, "status": STATUS_OK, "model": None}

In [5]:
best_run, _ = optim.minimize(
    model=create_model,
    data=create_data,
    algo=tpe.suggest,
    max_evals=200,
    trials=Trials(),
    notebook_name="notebooks/cifar10 hp-tuning vae"
)

>>> Imports:
#coding=utf-8

from __future__ import print_function

try:
    import os
except:
    pass

try:
    from hyperopt import STATUS_OK, Trials, tpe
except:
    pass

try:
    from hyperas.distributions import choice, uniform, loguniform
except:
    pass

try:
    from hyperas import optim
except:
    pass

try:
    import tensorflow as tf
except:
    pass

try:
    from tensorflow.keras import models, optimizers
except:
    pass

try:
    from tensorflow.keras.datasets import cifar10
except:
    pass

try:
    from sklearn.model_selection import train_test_split
except:
    pass

try:
    import numpy as np
except:
    pass

try:
    from autoencoder.variational_autoencoder import VariationalAutoencoder
except:
    pass

try:
    from common.utils import get_compile_args, get_callbacks, hyperas_path, i, best_acc, save_logs, init, load_samples, load_cifar10
except:
    pass

try:
    from sklearn.metrics import classification_report
except:
    pass

>>> Hyperas search space:



In [6]:
best_run

{'actv': 5,
 'beta': 2,
 'distribution': 0,
 'hidden_num': 2,
 'initialization': 1,
 'last_actv': 0,
 'loss_choice': 1,
 'lr': 3,
 'momentum': 0,
 'opt_choice': 5,
 'preprocess_choice': 2,
 'units1': 2,
 'units1_1': 0,
 'units1_2': 2,
 'units1_3': 0,
 'units1_4': 1,
 'use_batch_norm': 0,
 'weight_decay': 5}

In [7]:
from sklearn.metrics import classification_report


*_, clf = create_data() # preprocess="min-max"
model = models.load_model(os.path.join(hyperas_path, "cifar10_vae_00.tf"))
model.summary()

vae = VariationalAutoencoder(
    conditioned=True, 
    class_num=10,
    latent_dim=8,
    hiddens_dims=[64, 64, 16, 16],
    hiddens_kwargs={
        "actv": "sigmoid",
        "use_batch_norm": True,
        "kernel_init": "he_normal"
    },
    last_activation="linear", 
    beta=0.5,
    compile_args=get_compile_args(
        optimizer=optimizers.experimental.AdamW(learning_rate=3e-3, weight_decay=1e-2),
        loss="mean_absolute_error"
    ),
    name="cifar10_vae_00"
)

vae.set_weights(model.get_weights())
vae.seen_classes = list(range(10))

x_gen, y_true = vae.generate(onehot_labels=False)
y_preds = clf.predict(x_gen)
y_preds = np.argmax(y_preds, axis=-1)
print(classification_report(y_true, y_preds, digits=4))

Model: "cifar10_vae_00"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 encoder (Functional)        [(None, 8),               138000    
                              (None, 8),                         
                              (None, 8)]                         
                                                                 
 decoder (Functional)        (None, 2048)              139424    
                                                                 
Total params: 277,430
Trainable params: 276,784
Non-trainable params: 646
_________________________________________________________________
157/157 [==============================] - 0s 2ms/step
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       500
           1     1.0000    1.0000    1.0000       500
           2     1.0000    1.0000    1.0000       500
           3     0.9980    1.0000   